# Which electrode does the faulted station need?

A 20 kV cable feeder, an earth fault at a station in the middle, a fault
current with harmonics — and the question an earthing study often ends with:
which electrode does the faulted station need? This notebook

1. splits the network at the fault with `analyze_cuts` and separates what
   belongs to the **network** from what belongs to the **electrode**,
2. varies the electrode at the fault and watches both reduction factors,
3. runs a catalogue of electrode characteristics in the $\rho$–$f$ form
   through `run_sweep`, and
4. summarises and classifies the stacked results.

Every number is checked a second way: against the closed form of the ladder
network the feeder is, or against a nodal solve written here in NumPy. The
basics of cuts and reduction factors are in
[Decomposing a fault](../fault_decomposition/); the background in
[Splitting the network, location response](../../explanation/decomposition/).

In [ ]:
import warnings

import numpy as np
import polars as pl

import groundinsight as gi

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_width_chars(160)
print(f"groundinsight {gi.__version__}")

## The feeder

Six stations `S0 … S5` on a radial cable, 0.8 km per section, the infeed at
`S0` and the earth fault at `S3`. The source current carries the 50 Hz
fundamental and three harmonics. All values are illustrative:

| | value |
|---|---|
| station electrode | $10\ \Omega$, every station |
| shield–earth loop $Z_s'$ | $0.1 + 0.2j\,f/50\ \Omega$/km |
| coupling $Z_m'$ | $0.05 + 0.1j\,f/50\ \Omega$/km |
| phase–earth loop, shield bonded, $Z_\text{red}'$ | $0.06 + 0.11j\,f/50\ \Omega$/km |
| source current | 1000 A at 50 Hz, 120 A at 150 Hz, 60 A at 250 Hz, 30 A at 350 Hz |

The ratio $k = Z_m/Z_s = 0.5$ is the same on every section and at every
frequency.

In [ ]:
FREQUENCIES = [50.0, 150.0, 250.0, 350.0]
SOURCE = {50.0: 1000.0, 150.0: 120.0, 250.0: 60.0, 350.0: 30.0}   # A
Z_STATION = 10.0                                                  # Ohm
SECTION_KM = 0.8
BUSES = ["S0", "S1", "S2", "S3", "S4", "S5"]
SECTIONS = [("C01", "S0", "S1"), ("C12", "S1", "S2"), ("C23", "S2", "S3"),
            ("C34", "S3", "S4"), ("C45", "S4", "S5")]
FAULT_BUS = "S3"


def zs(f):
    "Shield-earth loop in Ohm/km."
    return 0.1 + 0.2j * f / 50


def zm(f):
    "Coupling phase conductor <-> shield loop in Ohm/km."
    return 0.05 + 0.1j * f / 50


STATION = gi.BusType(name="station", system_type="MV", voltage_level=20.0,
                     impedance_formula=repr(Z_STATION))
CABLE = gi.BranchType(name="cable", grounding_conductor=True,
                      self_impedance_formula="(0.1 + 0.2*j*f/50)*l",
                      mutual_impedance_formula="(0.05 + 0.1*j*f/50)*l",
                      phase_impedance_formula="(0.06 + 0.11*j*f/50)*l")


def feeder(sections=SECTIONS):
    net = gi.create_network(name="feeder", frequencies=FREQUENCIES)
    for bus in BUSES:
        gi.create_bus(name=bus, type=STATION, network=net)
    for name, a, b in sections:
        gi.create_branch(name=name, type=CABLE, from_bus=a, to_bus=b, length=SECTION_KM,
                         network=net)
    gi.create_source(name="infeed", bus="S0", values=SOURCE, network=net)
    gi.create_fault(name="earth_fault", bus=FAULT_BUS, network=net)
    return net


net = feeder()
gi.run_fault(net, fault="earth_fault")
net.res_fault("earth_fault").select("frequency_Hz", "is_rms", "I_F_A", "U_E_V", "Z_E_Ohm",
                                    "r_current", "r_epr")

### The second method: a nodal solve by hand

The earthing network is a ladder: electrodes to remote earth at every
station, the shields between them. Its nodal matrix takes ten lines. With the
same $k$ on every section, the phase current enters it only through $k$: the
earthing network is driven by $(1-k)\,I$, entering at the infeed and leaving
at the fault, in the sign frame of `groundinsight`.

In [ ]:
def impedance_matrix(f, electrode_at_fault=Z_STATION, sections=SECTIONS):
    "Inverse of the nodal admittance matrix of the earthing network at f."
    index = {bus: i for i, bus in enumerate(BUSES)}
    Y = np.zeros((len(BUSES), len(BUSES)), dtype=complex)
    for bus in BUSES:
        Y[index[bus], index[bus]] += 1 / (electrode_at_fault if bus == FAULT_BUS else Z_STATION)
    for _, a, b in sections:
        y = 1 / (zs(f) * SECTION_KM)
        i, j = index[a], index[b]
        Y[i, i] += y
        Y[j, j] += y
        Y[i, j] -= y
        Y[j, i] -= y
    return np.linalg.inv(Y), index


def u_e_by_hand(f, electrode_at_fault=Z_STATION, sections=SECTIONS):
    "EPR at the fault bus: (1 - k) I in at S0, out at S3."
    Z, index = impedance_matrix(f, electrode_at_fault, sections)
    k = zm(f) / zs(f)
    current = (1 - k) * SOURCE[f]
    return Z[index[FAULT_BUS], index["S0"]] * current - Z[index[FAULT_BUS], index[FAULT_BUS]] * current


res = net.res_fault("earth_fault").filter(~pl.col("is_rms"))
for row in res.iter_rows(named=True):
    f = row["frequency_Hz"]
    Z, index = impedance_matrix(f)
    print(f"{f:5.0f} Hz   U_E {row['U_E_V']:9.4f} V  by hand {abs(u_e_by_hand(f)):9.4f} V"
          f"   Z_E {row['Z_E_Ohm']:.6f} Ohm  by hand {abs(Z[index[FAULT_BUS], index[FAULT_BUS]]):.6f} Ohm")
rms = np.sqrt(sum(abs(u_e_by_hand(f)) ** 2 for f in FREQUENCIES))
print(f"  RMS   U_E {net.res_fault('earth_fault').filter(pl.col('is_rms'))['U_E_V'][0]:9.4f} V"
      f"  by hand {rms:9.4f} V")

## 1  The cuts: network and electrode apart

A `Cut` is a named set of branches at the fault bus. Here one points towards
the infeed and one towards the end of the feeder. `Z_side` is what the
network offers in that direction, `Z_local` is the electrode of the faulted
station itself, and `Z_parallel` combines them:
$1/Z_\text{parallel} = 1/Z_\text{local} + \sum 1/Z_\text{side}$.

In [ ]:
CUTS = [
    gi.Cut(name="towards_infeed", branches=["C23"], description="S3 -> S2 -> S1 -> S0"),
    gi.Cut(name="towards_end", branches=["C34"], description="S3 -> S4 -> S5, a passive tail"),
]
cuts = gi.analyze_cuts(net, fault="earth_fault", cuts=CUTS)
print("sides:", cuts.sides, "  disjoint:", cuts.sides_are_disjoint)

cuts.to_polars().filter(pl.col("frequency_Hz") == 50.0).select(
    "cut", "Z_side_Ohm", "Z_side_deg", "Z_local_Ohm", "Z_parallel_Ohm",
    "Z_driving_point_Ohm", "identity_residual",
)

On a radial feeder each side is a ladder, so $Z_\text{side}$ has a closed
form: the section, then the station electrode in parallel with the rest of
the ladder.

In [ ]:
def parallel(*impedances):
    return 1 / sum(1 / z for z in impedances)


def ladder(f, n_sections):
    "Seen from the fault bus through n sections, each ending at a 10 Ohm station."
    z = Z_STATION
    for _ in range(n_sections - 1):
        z = parallel(Z_STATION, zs(f) * SECTION_KM + z)
    return zs(f) * SECTION_KM + z


worst = 0.0
for f in FREQUENCIES:
    by_hand = {"towards_infeed": ladder(f, 3), "towards_end": ladder(f, 2)}
    z_par = parallel(Z_STATION, *by_hand.values())
    for name, z in by_hand.items():
        worst = max(worst, abs(cuts.z_side[name][f] - z) / abs(z))
    worst = max(worst, abs(cuts.z_parallel[f] - z_par) / abs(z_par))
    worst = max(worst, abs(cuts.z_driving_point[f] - z_par) / abs(z_par))
    print(f"{f:5.0f} Hz  Z_infeed = {by_hand['towards_infeed']:.5f}  Z_end = {by_hand['towards_end']:.5f}"
          f"  Z_parallel = {z_par:.5f} Ohm")
print(f"\nlargest relative deviation of groundinsight from the closed forms: {worst:.1e}")

The currents at the fault bus close as well: the electrode current and the
shield currents of the two cuts add up to the fault current.

In [ ]:
frame = cuts.to_polars()
print("largest KCL residual at the fault bus:", f"{frame['kcl_residual'].max():.1e}")
frame.filter(pl.col("frequency_Hz") == 50.0).select(
    "cut", "I_fault_A", "I_local_A", "I_shield_A", "current_share", "I_earth_A", "r_side",
)

`r_side` is left empty on the passive tail: no source current enters the
network from that direction, so there is no share to take a ratio of.

## 2  The electrode at the fault, varied

The same feeder with five electrodes at `S3`, from $0.5$ to $200\ \Omega$.
Each one is a `SweepPoint` that overrides the impedance table of the bus;
`run_sweep` restores the network afterwards.

In [ ]:
ELECTRODES = [0.5, 2.0, 10.0, 50.0, 200.0]
points = [
    gi.SweepPoint(label=f"Z_B={z:g}", bus_impedance={FAULT_BUS: {f: complex(z) for f in FREQUENCIES}},
                  parameters={"Z_B_Ohm": z})
    for z in ELECTRODES
]
with warnings.catch_warnings():
    warnings.simplefilter("ignore")      # every point keeps its diagnostics
    electrodes = gi.run_sweep(net, fault="earth_fault", points=points, cuts=CUTS)
print("failures:", electrodes.failures or "none")

fifty = pl.col("frequency_Hz") == 50.0
u_e = (electrodes.buses().filter(fifty & (pl.col("bus_name") == FAULT_BUS))
       .select("point", pl.col("EPR_V").alias("U_E_V")))
sides = (electrodes.cuts().filter(fifty)
         .pivot(on="cut", index="point", values="Z_side_Ohm"))
table = (electrodes.impedances().filter(fifty)
         .select("point", "Z_B_Ohm", "grounding_impedance_Ohm", "reduction_factor",
                 "reduction_factor_current")
         .join(u_e, on="point").join(sides, on="point"))
table

In [ ]:
worst = 0.0
for row in table.iter_rows(named=True):
    by_hand = abs(u_e_by_hand(50.0, electrode_at_fault=row["Z_B_Ohm"]))
    worst = max(worst, abs(row["U_E_V"] - by_hand) / by_hand)
print(f"largest relative deviation of U_E from the nodal solve by hand: {worst:.1e}")

Three kinds of quantity, three behaviours:

* `Z_side` does not move. It belongs to the network, not to the electrode.
* $Z_E$ (`grounding_impedance_Ohm`), $U_E$ and the reduction factor on a
  current basis, `reduction_factor_current`, follow the electrode.
* The EPR ratio, `reduction_factor`, stays at $0.5 = \lvert 1 - k\rvert$ over
  two and a half decades of $Z_B$. It is computed from two solves that share
  the same nodal matrix, so a change at the fault bus cancels in the
  quotient — which is why the EN 50522 reduction factor of
  [Reduction factors and earthing impedance](../../explanation/reduction-factors/)
  is the current-based one.

## 3  A catalogue of electrodes in the $\rho$–$f$ form

A field computation or a measurement campaign can hand an electrode over as
five numbers $k_1 \dots k_5$ of the form

$$
Z(\rho, f) = k_1\,\rho + (k_2 + j\,k_3)\,f + (k_4 + j\,k_5)\,\rho\,f .
$$

`rho_f_points` turns a catalogue of such vectors into sweep points for one
bus, evaluated at the network frequencies and a soil resistivity, and copies
$k_1 \dots k_5$ and $\rho$ into the result frames as ordinary columns.

In [ ]:
RHO = 100.0   # Ohm m
CATALOGUE = {
    "very good": (0.005, 1.0e-4, 3.0e-4, 0.0, 0.0),
    "good":      (0.020, 1.0e-4, 3.0e-4, 0.0, 0.0),
    "medium":    (0.050, 1.0e-4, 3.0e-4, 0.0, 0.0),
    "poor":      (0.100, 1.0e-4, 3.0e-4, 0.0, 0.0),
    "very poor": (0.200, 1.0e-4, 3.0e-4, 0.0, 0.0),
}
points = gi.rho_f_points(bus=FAULT_BUS, k_vectors=CATALOGUE, frequencies=FREQUENCIES, rho=RHO)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    catalogue = gi.run_sweep(net, fault="earth_fault", points=points, cuts=CUTS)
print("solved:", catalogue.labels, "  failures:", catalogue.failures or "none")

catalogue.cuts().filter(pl.col("frequency_Hz") == 50.0).select(
    "point", "k1", "cut", "Z_side_Ohm", "Z_local_Ohm", "Z_parallel_Ohm", "current_share",
)

In [ ]:
def form(k, f, rho=RHO):
    k1, k2, k3, k4, k5 = k
    return k1 * rho + (k2 + 1j * k3) * f + (k4 + 1j * k5) * rho * f


cut_frame = catalogue.cuts().filter(pl.col("cut") == "towards_infeed")
bus_frame = catalogue.buses().filter((pl.col("bus_name") == FAULT_BUS) & ~pl.col("is_rms"))
dev_local = dev_parallel = dev_u = 0.0
for label, k in CATALOGUE.items():
    for f in FREQUENCIES:
        z_b = form(k, f)
        row = cut_frame.filter((pl.col("point") == label) & (pl.col("frequency_Hz") == f)).row(0, named=True)
        z_par = parallel(z_b, ladder(f, 3), ladder(f, 2))
        u = bus_frame.filter((pl.col("point") == label) & (pl.col("frequency_Hz") == f))["EPR_V"][0]
        dev_local = max(dev_local, abs(row["Z_local_Ohm"] - abs(z_b)) / abs(z_b))
        dev_parallel = max(dev_parallel, abs(row["Z_parallel_Ohm"] - abs(z_par)) / abs(z_par))
        u_hand = abs(u_e_by_hand(f, electrode_at_fault=z_b))
        dev_u = max(dev_u, abs(u - u_hand) / u_hand)
print(f"Z_local    against the form by hand:          {dev_local:.1e}")
print(f"Z_parallel against the ladder closed forms:   {dev_parallel:.1e}")
print(f"U_E        against the nodal solve by hand:   {dev_u:.1e}")

Every number of the sweep — 5 electrodes × 4 frequencies — agrees with the
closed forms and the nodal solve to rounding.

## 4  Statistics and classification

The sweep frames are long-format tables, one row per point, bus and
frequency plus an RMS row, so the usual Polars tools apply. `summarize`
reduces a column to count, spread, quantiles and extremes, `classify` bins
it into bands that **you** supply — there is deliberately no built-in table
of admissible values. The edges below are illustrative, not limits of a
standard.

In [ ]:
rms = catalogue.buses().filter(pl.col("is_rms"))
summary = gi.summarize(rms, "EPR_V", by=["bus_name"]).sort("bus_name")
summary

In [ ]:
by_hand = rms.group_by("bus_name").agg(
    pl.col("EPR_V").mean().alias("mean"), pl.col("EPR_V").median().alias("p50"),
    pl.col("EPR_V").max().alias("max")).sort("bus_name")
worst = max(float((summary[c] - by_hand[c]).abs().max()) for c in ("mean", "p50", "max"))
print(f"summarize against a plain group_by: largest difference {worst:.1e} V")

In [ ]:
EDGES = [30.0, 60.0]                       # V, illustrative
banded = gi.classify(rms, "EPR_V", EDGES, labels=["below 30 V", "30 to 60 V", "above 60 V"])
print(banded.filter(pl.col("bus_name") == FAULT_BUS).select("point", "k1", "EPR_V", "class").sort("k1"))

counts = banded.group_by("class").len().sort("class")
by_hand = {
    "below 30 V": rms.filter(pl.col("EPR_V") < 30.0).height,
    "30 to 60 V": rms.filter((pl.col("EPR_V") >= 30.0) & (pl.col("EPR_V") < 60.0)).height,
    "above 60 V": rms.filter(pl.col("EPR_V") >= 60.0).height,
}
print()
print("rows per class:", dict(counts.iter_rows()), "  by hand:", by_hand)

## 5  When the feeder is closed into a ring

A sixth section `S5 – S0` closes the feeder into a ring. The two cuts no
longer lead to separate parts of the network: both directions end up at the
infeed. The decomposition stays exact — `Z_side` is defined by how a test
current injected at the fault bus divides, and the identity holds whatever
the topology — but the two sides share their stations, `sides_are_disjoint`
is `False`, and `r_side`, which needs a side of its own, is left empty.

In [ ]:
RING = SECTIONS + [("C50", "S5", "S0")]
ring = feeder(RING)
gi.run_fault(ring, fault="earth_fault")
ring_cuts = gi.analyze_cuts(ring, fault="earth_fault", cuts=CUTS)
print("sides:", ring_cuts.sides, "  disjoint:", ring_cuts.sides_are_disjoint)

# the same test current, by hand: 1 A into S3, the share through each cut
Z, index = impedance_matrix(50.0, sections=RING)
u = Z[:, index[FAULT_BUS]]
for name, neighbour in (("towards_infeed", "S2"), ("towards_end", "S4")):
    i_cut = (u[index[FAULT_BUS]] - u[index[neighbour]]) / (zs(50.0) * SECTION_KM)
    z_hand = u[index[FAULT_BUS]] / i_cut
    print(f"{name:15s} Z_side {abs(ring_cuts.z_side[name][50.0]):.6f} Ohm, by hand {abs(z_hand):.6f} Ohm,"
          f"  r_side {ring_cuts.r_side[name][50.0]}")
print(f"identity residual {max(ring_cuts.identity_residual.values()):.1e}")

## Summary

| quantity | what it describes | moves with the electrode at the fault |
|---|---|---|
| `Z_side_Ohm` per cut | what the network offers in that direction | no — a property of the network |
| `Z_local_Ohm` | the electrode of the faulted station | it is the electrode |
| `Z_parallel_Ohm` | both in parallel, equal to $Z_E$ | yes |
| `reduction_factor` (EPR ratio) | EPR with coupling over EPR without | **no** |
| `reduction_factor_current` | share of the fault current returning through the soil | yes |
| `current_share` per cut | share of the fault current leaving through that cut | yes |
| `r_side` | earth share over that cut | yes, and only for disjoint sides with a source |

The API used here: [Fault-location decomposition](../../api/decomposition/),
[Parameter sweeps](../../api/sweep/),
[Statistics and classification](../../api/statistics/).